# Forest continuity hexagon maps

This notebook is reduced to the hexagon workflow only. It loads the binary forest-continuity raster, the forest mask and the country geometries, then creates or loads a cached hexagon GeoPackage and plots two maps:

1. relative forest-continuity share per hexagon, in percent of forest area. The colorbar starts at 0.1%;
2. absolute forest-continuity area per hexagon, in km².

NBR trend, genus composition, probability-by-genus plots, country summaries and CSO inset maps are intentionally left out for now.

The mainland clipping extent is expanded to **45°E**. Ukraine is kept in the country list, so the eastern part of the study area should not be cut at the former 35°E boundary.

In [ ]:
# ------------------------------------------------------------
# Imports and global settings
# ------------------------------------------------------------
from pathlib import Path
from collections import Counter
from datetime import datetime
import shutil
import tempfile

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.vrt import WarpedVRT
from rasterio.enums import Resampling
from rasterio.features import rasterize
from rasterio.windows import bounds as window_bounds
from rasterio.windows import transform as window_transform

import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.patches import Patch
from shapely.geometry import Polygon, box
from tqdm.auto import tqdm

# ------------------------------------------------------------
# Main inputs
# ------------------------------------------------------------
# Binary forest-continuity product. Pixels with value 1 are treated as continuously undisturbed forest.
continuity_fp = Path(
    "/mnt/eo/EO4Backcasting/_predictions/predictions_global_model/"
    "global_model_no_dist_gt08_4conn_5ha_TRUE_0_1.tif"
)

# Forest mask aligned or alignable to the continuity raster. Pixels with value 1 are forest.
forest_mask_fp = Path("/mnt/eo/EFDA_v211/forest_landuse_aligned.tif")

# Country polygons in EPSG:3035.
country_gpkg = Path("/mnt/eo/EO4Backcasting/_data/CNTR_RG_10M_2024_3035.gpkg")
country_name_col = "NAME_ENGL"

# ------------------------------------------------------------
# Output/cache folders
# ------------------------------------------------------------
out_dir = Path("/mnt/eo/EO4Backcasting/figures")
out_dir.mkdir(parents=True, exist_ok=True)

cache_dir = Path("/mnt/eo/EO4Backcasting/_analysis_cache")
cache_dir.mkdir(parents=True, exist_ok=True)

# Use a new cache name so an older product clipped at 35°E is not silently reused.
hex_gpkg = cache_dir / "forest_continuity_by_small_clipped_hexagon_lon45.gpkg"

FORCE_RECOMPUTE = False

# ------------------------------------------------------------
# Analysis settings
# ------------------------------------------------------------
pixel_area_ha = 0.09       # 30 m pixel = 900 m² = 0.09 ha
forest_value = 1
continuity_value = 1

hex_radius = 17_500        # metres; approximately 500 km² hexagons
min_forest_area_ha = 500   # keep only hexagons with at least this much forest area

# Mainland Europe extent used to remove remote/overseas parts and avoid the old 35°E cutoff.
# 45°E keeps the eastern part of Ukraine within the processing extent.
mainland_bbox_4326 = box(-11, 34, 45, 72)

# A product with max longitude below this value is very likely still based on the old 35°E clipping.
eastern_extent_check_min_lon = 38.0

# Countries to keep. Ukraine is intentionally included.
european_countries = [
    "Albania", "Andorra", "Austria", "Belarus", "Belgium",
    "Bosnia and Herzegovina", "Bulgaria", "Croatia", "Czechia",
    "Denmark", "Estonia", "Finland", "France", "Germany",
    "Greece", "Hungary", "Ireland", "Italy", "Kosovo", "Latvia",
    "Liechtenstein", "Lithuania", "Luxembourg", "Moldova", "Monaco",
    "Montenegro", "Netherlands", "North Macedonia", "Norway", "Poland",
    "Portugal", "Romania", "San Marino", "Serbia", "Slovakia",
    "Slovenia", "Spain", "Sweden", "Switzerland", "Ukraine",
    "United Kingdom", "Vatican"
]

excluded_countries = {"Russian Federation", "Russia", "Turkey", "Kazakhstan"}


In [ ]:
# ------------------------------------------------------------
# Helper functions
# ------------------------------------------------------------
def create_hex_grid(bounds, hex_radius, crs):
    """Create a flat-top hexagon grid for a given bounding box."""
    minx, miny, maxx, maxy = bounds
    dx = 1.5 * hex_radius
    dy = np.sqrt(3) * hex_radius

    hexagons = []
    hex_ids = []
    x = minx - 2 * hex_radius
    col = 0
    hex_id = 1

    while x < maxx + 2 * hex_radius:
        y_offset = 0 if col % 2 == 0 else dy / 2
        y = miny - dy

        while y < maxy + dy:
            cy = y + y_offset
            angles = np.deg2rad([0, 60, 120, 180, 240, 300])
            coords = [(x + hex_radius * np.cos(a), cy + hex_radius * np.sin(a)) for a in angles]
            hexagons.append(Polygon(coords))
            hex_ids.append(hex_id)
            hex_id += 1
            y += dy

        x += dx
        col += 1

    return gpd.GeoDataFrame({"hex_id": hex_ids}, geometry=hexagons, crs=crs)


def geometry_union(geoseries):
    """Compatibility wrapper for GeoPandas versions with or without union_all()."""
    if hasattr(geoseries, "union_all"):
        return geoseries.union_all()
    return geoseries.unary_union


def load_europe_countries(target_crs=None, clip_mainland=True):
    """Load European country polygons and optionally clip them to mainland Europe up to 45°E."""
    countries = gpd.read_file(country_gpkg)
    countries["country"] = countries[country_name_col].astype(str).str.strip()

    countries = countries[countries["country"].isin(european_countries)].copy()
    countries = countries[~countries["country"].isin(excluded_countries)].copy()

    countries = countries[countries.geometry.notna() & ~countries.geometry.is_empty].copy()
    countries["geometry"] = countries.geometry.make_valid()

    if clip_mainland:
        countries_4326 = countries.to_crs(4326)
        parts = countries_4326.explode(index_parts=False).reset_index(drop=True)

        minx, miny, maxx, maxy = mainland_bbox_4326.bounds
        representative_points = parts.representative_point()
        keep = (
            (representative_points.x >= minx) & (representative_points.x <= maxx) &
            (representative_points.y >= miny) & (representative_points.y <= maxy)
        )
        parts = parts.loc[keep].copy()

        bbox_gdf = gpd.GeoDataFrame(geometry=[mainland_bbox_4326], crs=4326)
        parts = gpd.clip(parts, bbox_gdf)
        countries = parts.dissolve(by="country", as_index=False)

    if target_crs is not None and countries.crs != target_crs:
        countries = countries.to_crs(target_crs)

    return countries


def cache_is_stale_or_clipped(gdf, min_expected_max_lon=eastern_extent_check_min_lon):
    """Return True if cached hexagons still look clipped at the old 35°E extent."""
    max_lon = gdf.to_crs(4326).total_bounds[2]
    return max_lon < min_expected_max_lon


def backup_existing_gpkg(path):
    """Rename an existing GeoPackage and possible SQLite sidecar files with a timestamp."""
    path = Path(path)

    if not path.exists():
        return

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    backup_path = path.with_name(f"{path.stem}_backup_{timestamp}{path.suffix}")
    path.rename(backup_path)
    print(f"Existing GPKG renamed to: {backup_path}")

    for suffix in ["-wal", "-shm", "-journal"]:
        sidecar = Path(str(path) + suffix)
        if sidecar.exists():
            sidecar_backup = path.with_name(f"{path.stem}_backup_{timestamp}{path.suffix}{suffix}")
            sidecar.rename(sidecar_backup)
            print(f"Existing sidecar renamed to: {sidecar_backup}")


def choose_tmp_base():
    """Choose a temporary write folder for GeoPackage creation."""
    candidates = [
        Path("/mnt/dss_project/lmandl/tmp_gpkg_write"),
        Path(tempfile.gettempdir()) / "lmandl_tmp_gpkg_write",
    ]

    for candidate in candidates:
        try:
            candidate.mkdir(parents=True, exist_ok=True)
            return candidate
        except Exception:
            continue

    raise RuntimeError("No writable temporary directory found for GeoPackage writing.")


def write_hexes_gpkg_with_backup(gdf, path, layer="hexes"):
    """
    Write the GeoDataFrame to a temporary path first, validate it,
    then copy the finished GeoPackage to the final target path.
    This avoids common SQLite/GPKG transaction errors on network mounts.
    """
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    gdf_out = gdf.copy()
    gdf_out = gdf_out[gdf_out.geometry.notna() & ~gdf_out.geometry.is_empty].copy()
    gdf_out["geometry"] = gdf_out.geometry.make_valid()

    tmp_base = choose_tmp_base()

    with tempfile.TemporaryDirectory(dir=tmp_base) as tmp_dir:
        tmp_path = Path(tmp_dir) / path.name
        print(f"Writing temporary GPKG first: {tmp_path}")

        try:
            gdf_out.to_file(tmp_path, layer=layer, driver="GPKG", engine="pyogrio")
        except Exception as e_pyogrio:
            print("pyogrio write failed. Trying fiona instead.")
            print(f"pyogrio error: {e_pyogrio}")
            gdf_out.to_file(tmp_path, layer=layer, driver="GPKG", engine="fiona")

        test = gpd.read_file(tmp_path, layer=layer)
        print(f"Temporary GPKG successfully written and read: {len(test)} features")

        backup_existing_gpkg(path)
        shutil.copy2(tmp_path, path)

    print(f"Written final GPKG: {path}")


def try_read_cached_hexes(path, layer="hexes"):
    """Try to read a cached hexagon product. Return None if it is missing or invalid."""
    path = Path(path)

    if not path.exists():
        return None

    try:
        gdf = gpd.read_file(path, layer=layer)
        print(f"Loaded cached hexagon product: {path}")
        return gdf
    except Exception as e:
        print("Existing GPKG could not be read and will be recomputed.")
        print(f"Read error: {e}")
        backup_existing_gpkg(path)
        return None


In [ ]:
# ------------------------------------------------------------
# Load or compute hexagon summary
# ------------------------------------------------------------
def compute_hexagon_summary():
    with rasterio.open(continuity_fp) as continuity_src:
        countries_eur = load_europe_countries(
            target_crs=continuity_src.crs,
            clip_mainland=True
        )

    europe_geom = geometry_union(countries_eur.geometry)
    europe_bounds = countries_eur.total_bounds

    hexes = create_hex_grid(
        bounds=europe_bounds,
        hex_radius=hex_radius,
        crs=countries_eur.crs
    )

    hexes = hexes[hexes.intersects(europe_geom)].copy()

    europe_clip = gpd.GeoDataFrame(geometry=[europe_geom], crs=countries_eur.crs)
    hexes = gpd.clip(hexes, europe_clip)
    hexes = hexes[hexes.geometry.notna() & ~hexes.geometry.is_empty].copy()
    hexes["geometry"] = hexes.geometry.make_valid()
    hexes = hexes.reset_index(drop=True)
    hexes["hex_id"] = np.arange(1, len(hexes) + 1).astype(np.int32)

    print("Number of clipped hexagons:", len(hexes))

    hex_sindex = hexes.sindex
    forest_counter = Counter()
    continuity_counter = Counter()

    with rasterio.open(continuity_fp) as continuity_src, rasterio.open(forest_mask_fp) as forest_src:
        with WarpedVRT(
            forest_src,
            crs=continuity_src.crs,
            transform=continuity_src.transform,
            width=continuity_src.width,
            height=continuity_src.height,
            resampling=Resampling.nearest,
            src_nodata=forest_src.nodata,
            nodata=0,
        ) as forest_vrt:

            windows = [window for _, window in continuity_src.block_windows(1)]

            for window in tqdm(
                windows,
                desc="Counting forest and forest-continuity area by clipped hexagon"
            ):
                continuity = continuity_src.read(1, window=window)
                forest = forest_vrt.read(1, window=window)

                if not (np.any(forest == forest_value) or np.any(continuity == continuity_value)):
                    continue

                block_geom = box(*window_bounds(window, continuity_src.transform))

                candidate_idx = list(hex_sindex.intersection(block_geom.bounds))
                if not candidate_idx:
                    continue

                hex_block = hexes.iloc[candidate_idx].copy()
                hex_block = hex_block[hex_block.intersects(block_geom)]

                if hex_block.empty:
                    continue

                hex_raster = rasterize(
                    shapes=zip(hex_block.geometry, hex_block["hex_id"]),
                    out_shape=continuity.shape,
                    transform=window_transform(window, continuity_src.transform),
                    fill=0,
                    dtype="int32",
                    all_touched=False,
                )

                valid_hex = hex_raster != 0
                valid_forest = valid_hex & (forest == forest_value)

                forest_ids = hex_raster[valid_forest].astype(np.int32)
                continuity_ids = hex_raster[
                    valid_forest & (continuity == continuity_value)
                ].astype(np.int32)

                if forest_ids.size > 0:
                    unique, counts = np.unique(forest_ids, return_counts=True)
                    forest_counter.update(dict(zip(unique, counts)))

                if continuity_ids.size > 0:
                    unique, counts = np.unique(continuity_ids, return_counts=True)
                    continuity_counter.update(dict(zip(unique, counts)))

    df_forest_hex = pd.DataFrame(
        forest_counter.items(),
        columns=["hex_id", "forest_pixels"]
    )

    df_continuity_hex = pd.DataFrame(
        continuity_counter.items(),
        columns=["hex_id", "continuity_pixels"]
    )

    df_hex = df_forest_hex.merge(
        df_continuity_hex,
        on="hex_id",
        how="left"
    )

    df_hex["continuity_pixels"] = df_hex["continuity_pixels"].fillna(0)
    df_hex["forest_area_ha"] = df_hex["forest_pixels"] * pixel_area_ha
    df_hex["continuity_area_ha"] = df_hex["continuity_pixels"] * pixel_area_ha
    df_hex["continuity_area_km2"] = df_hex["continuity_area_ha"] / 100
    df_hex["continuity_share_percent"] = (
        df_hex["continuity_area_ha"] / df_hex["forest_area_ha"] * 100
    )

    hexes_plot = hexes.merge(df_hex, on="hex_id", how="left")
    hexes_plot = hexes_plot[
        hexes_plot["forest_area_ha"].notna() &
        (hexes_plot["forest_area_ha"] >= min_forest_area_ha)
    ].copy()

    return hexes_plot, countries_eur


hexes_plot = None
countries_eur = None

if hex_gpkg.exists() and not FORCE_RECOMPUTE:
    hexes_cached = try_read_cached_hexes(hex_gpkg, layer="hexes")

    if hexes_cached is not None and cache_is_stale_or_clipped(hexes_cached):
        print(
            f"Cached hexagon product appears clipped at "
            f"{hexes_cached.to_crs(4326).total_bounds[2]:.2f}°E. Recomputing."
        )
        hexes_cached = None

    if hexes_cached is not None:
        hexes_plot = hexes_cached
        with rasterio.open(continuity_fp) as src:
            countries_eur = load_europe_countries(target_crs=src.crs, clip_mainland=True)
    else:
        hexes_plot, countries_eur = compute_hexagon_summary()
        write_hexes_gpkg_with_backup(hexes_plot, hex_gpkg, layer="hexes")
else:
    if FORCE_RECOMPUTE:
        print("FORCE_RECOMPUTE=True. Recomputing hexagon product.")
    hexes_plot, countries_eur = compute_hexagon_summary()
    write_hexes_gpkg_with_backup(hexes_plot, hex_gpkg, layer="hexes")

hex_bounds_4326 = hexes_plot.to_crs(4326).total_bounds
country_bounds_4326 = countries_eur.to_crs(4326).total_bounds

print("Hexagons with sufficient forest:", len(hexes_plot))
print("Hex bounds EPSG:4326:", hex_bounds_4326)
print("Country bounds EPSG:4326:", country_bounds_4326)

if hex_bounds_4326[2] < eastern_extent_check_min_lon:
    raise RuntimeError(
        "The hexagon product still appears to be clipped too far west. "
        f"Current max longitude is {hex_bounds_4326[2]:.2f}°E. "
        "Check the country selection, mainland_bbox_4326 and whether an old cache was reused."
    )

display(
    hexes_plot[
        [
            "hex_id",
            "forest_area_ha",
            "continuity_area_ha",
            "continuity_area_km2",
            "continuity_share_percent"
        ]
    ].head()
)


In [ ]:
# ------------------------------------------------------------
# Plot helper
# ------------------------------------------------------------
def add_country_boundaries(ax, countries, target_crs):
    countries_plot = countries.to_crs(target_crs)
    countries_plot.boundary.plot(
        ax=ax,
        linewidth=0.40,
        edgecolor="0.30",
        alpha=0.85,
        zorder=10,
    )


def save_figure(fig, png_path, svg_path):
    fig.savefig(png_path, dpi=300, bbox_inches="tight", pad_inches=0.02, facecolor="white")
    fig.savefig(svg_path, bbox_inches="tight", pad_inches=0.02, facecolor="white")
    print("Saved PNG:", png_path)
    print("Saved SVG:", svg_path)


In [ ]:
# ------------------------------------------------------------
# Plot 1: Relative forest-continuity share by hexagon
# Values < 0.1% are shown in very light grey.
# The colorbar starts at 0.1% and is fixed to 0.1–7%; values above 7% use the darkest color.
# No separate legend box is added for values < 0.1%.
# ------------------------------------------------------------
rel_col = "continuity_share_percent"
low_threshold = 0.1

hexes_rel = hexes_plot[hexes_plot[rel_col].notna()].copy()
hexes_low = hexes_rel[hexes_rel[rel_col] < low_threshold].copy()
hexes_high = hexes_rel[hexes_rel[rel_col] >= low_threshold].copy()

print(hexes_rel[rel_col].describe())
print(f"Hexagons with {rel_col} < {low_threshold}: {len(hexes_low)}")
print(f"Hexagons with {rel_col} >= {low_threshold}: {len(hexes_high)}")

vmin = low_threshold
vmax = 7
norm = mpl.colors.Normalize(vmin=vmin, vmax=vmax)

cmap = plt.get_cmap("YlGnBu").copy()
cmap.set_bad(alpha=0)

fig, ax = plt.subplots(figsize=(10, 10))

if len(hexes_low) > 0:
    hexes_low.plot(
        ax=ax,
        color="0.94",
        linewidth=0.18,
        edgecolor="0.82",
        alpha=1.0,
        zorder=1,
    )

if len(hexes_high) > 0:
    hexes_high.plot(
        column=rel_col,
        ax=ax,
        cmap=cmap,
        norm=norm,
        linewidth=0.18,
        edgecolor="0.82",
        alpha=1.0,
        zorder=2,
    )

add_country_boundaries(ax, countries_eur, hexes_rel.crs)

sm = mpl.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])

cbar = fig.colorbar(
    sm,
    ax=ax,
    shrink=0.72,
    pad=0.02,
    ticks=[low_threshold, 1, 2, 3, 4, 5, 6, 7],
    extend="max",
)
cbar.ax.set_yticklabels(["0.1", "1", "2", "3", "4", "5", "6", ">7"])
cbar.set_label("Forest-continuity share of forest area (%)", fontsize=12)
cbar.ax.tick_params(labelsize=12)
cbar.outline.set_linewidth(0.6)

ax.set_axis_off()

out_png_rel = out_dir / "forest_continuity_share_hexagons_linear_0_1_7_lowvals_grey_countries_lon45.png"
out_svg_rel = out_dir / "forest_continuity_share_hexagons_linear_0_1_7_lowvals_grey_countries_lon45.svg"

save_figure(fig, out_png_rel, out_svg_rel)
plt.show()


In [ ]:
# ------------------------------------------------------------
# Plot 2: Absolute forest-continuity area by hexagon
# Values equal to 0 km² are shown in very light grey.
# The color scale is capped at the 98th percentile to avoid one or two large hexagons dominating the map.
# ------------------------------------------------------------
abs_col = "continuity_area_km2"
zero_threshold = 0

hexes_abs = hexes_plot[hexes_plot[abs_col].notna()].copy()
hexes_zero = hexes_abs[hexes_abs[abs_col] <= zero_threshold].copy()
hexes_positive = hexes_abs[hexes_abs[abs_col] > zero_threshold].copy()

print(hexes_abs[abs_col].describe())
print(f"Hexagons with {abs_col} <= {zero_threshold}: {len(hexes_zero)}")
print(f"Hexagons with {abs_col} > {zero_threshold}: {len(hexes_positive)}")

if len(hexes_positive) == 0:
    raise ValueError("No hexagons with positive forest-continuity area found.")

vmin_abs = 0
vmax_abs = float(np.nanpercentile(hexes_positive[abs_col], 98))
if vmax_abs <= 0:
    vmax_abs = float(hexes_positive[abs_col].max())

norm_abs = mpl.colors.Normalize(vmin=vmin_abs, vmax=vmax_abs)

cmap_abs = plt.get_cmap("YlGnBu").copy()
cmap_abs.set_bad(alpha=0)

fig, ax = plt.subplots(figsize=(10, 10))

if len(hexes_zero) > 0:
    hexes_zero.plot(
        ax=ax,
        color="0.94",
        linewidth=0.18,
        edgecolor="0.82",
        alpha=1.0,
        zorder=1,
    )

hexes_positive.plot(
    column=abs_col,
    ax=ax,
    cmap=cmap_abs,
    norm=norm_abs,
    linewidth=0.18,
    edgecolor="0.82",
    alpha=1.0,
    zorder=2,
)

add_country_boundaries(ax, countries_eur, hexes_abs.crs)

sm_abs = mpl.cm.ScalarMappable(cmap=cmap_abs, norm=norm_abs)
sm_abs.set_array([])

cbar = fig.colorbar(
    sm_abs,
    ax=ax,
    shrink=0.72,
    pad=0.02,
    extend="max",
)
cbar.set_label("Forest-continuity area per hexagon (km²)", fontsize=12)
cbar.ax.tick_params(labelsize=12)
cbar.outline.set_linewidth(0.6)

legend_handles = [
    Patch(
        facecolor="0.94",
        edgecolor="0.82",
        label="0 km²",
    )
]

ax.legend(
    handles=legend_handles,
    loc="lower left",
    frameon=True,
    framealpha=0.9,
    fontsize=11,
)

ax.set_axis_off()

out_png_abs = out_dir / "forest_continuity_area_hexagons_p98cap_zero_grey_countries_lon45.png"
out_svg_abs = out_dir / "forest_continuity_area_hexagons_p98cap_zero_grey_countries_lon45.svg"

save_figure(fig, out_png_abs, out_svg_abs)
plt.show()
